# Retriever

## Installing Haystack

In [ ]:
%%bash

pip install --upgrade pip
pip install 'farm-haystack[faiss]'

## Logging

We configure how logging messages should be displayed and which log level should be used before importing Haystack.
Example log message:
INFO - haystack.utils.preprocessing -  Converting data/tutorial1/218_Olenna_Tyrell.txt
Default log level in basicConfig is WARNING so the explicit parameter is not necessary but can be changed easily:

In [1]:
import logging

logging.basicConfig(format="%(levelname)s - %(name)s -  %(message)s", level=logging.WARNING)
logging.getLogger("haystack").setLevel(logging.INFO)

### Initialize Document Store - FAISS

FAISS is a library for efficient similarity search on a cluster of dense vectors.
The `FAISSDocumentStore` uses a SQL(SQLite in-memory be default) database under-the-hood
to store the document text and other meta data. The vector embeddings of the text are
indexed on a FAISS Index that later is queried for searching answers.
The default flavour of FAISSDocumentStore is "Flat" but can also be set to "HNSW" for
faster search at the expense of some accuracy. Just set the faiss_index_factor_str argument in the constructor.
For more info on which suits your use case: https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index

In [1]:
from haystack.document_stores import FAISSDocumentStore

document_store = FAISSDocumentStore(faiss_index_factory_str="Flat")

### Cleaning & indexing documents

We preprocess our data and use the sliding window technique to split the documents.

In [19]:
# Import the necessary modules and functions from Haystack
from haystack.utils import convert_files_to_docs
from haystack.nodes import PreProcessor, TextConverter, PDFToTextConverter, DocxToTextConverter

# Converters are used to convert documents into text
# Here we are using a PDFToTextConverter to convert PDF documents to text
converter = PDFToTextConverter(
    remove_numeric_tables=True,  # If True, removes tables that consist of only numbers
    #valid_languages=["de"]      # Only documents in German ("deu") should be converted
)

# Use the converter to convert a specific PDF file into text
doc_pdf = converter.convert(
    file_path="/Users/svenschnydrig/Library/CloudStorage/OneDrive-SharedLibraries-UniversitätSt.Gallen/STUD-NLP Group Project - General/Project Code/data/main.pdf", meta=None)[0]  # Metadata for the document. If None, no metadata will be added
# The convert method returns a list of documents, we're only interested in the first one

""""
# PreProcessor is used to perform preprocessing on the text
preprocessor_sliding_window = PreProcessor(
    #split_overlap=150,  # Number of words that the resulting chunks will overlap
    #split_length=400,   # Number of words in one chunk
    #split_respect_sentence_boundary=False,  # If True, splits would be made at sentence boundaries
    clean_empty_lines=True,  # If True, normalizes 3 or more consecutive empty lines to be just two empty lines
    clean_whitespace=True,  # If True, removes any whitespace at the beginning or end of each line in the text
    clean_header_footer=False,  # If True, removes any long header or footer texts that are repeated on each page
    #split_by="word",  # Document is split on word boundaries
    language="de",  # Language of the document
)
"""

# Use the preprocessor to process the converted document
#docs_sliding_window = preprocessor_sliding_window.process([doc_pdf])

# Write the processed documents to a Document Store
# Please note that the document_store object must be already initialized in the scope
document_store.write_documents(docs_sliding_window)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


pdftotext version 4.04 [www.xpdfreader.com]
Copyright 1996-2022 Glyph & Cog, LLC


Preprocessing:   0%|          | 0/1 [00:00<?, ?docs/s]

Writing Documents:   0%|          | 0/249 [00:00<?, ?it/s]

# Initialize Retriever, Reader & Pipeline

## Retriever

We decided to use an `EmbeddingRetriever` and `DensePassageRetriever`.

**Alternatives:**

- `BM25Retriever` with custom queries (for example, boosting) and filters
- `TfidfRetriever` in combination with a SQL or InMemory Document store for simple prototyping and debugging

### DPR

In [20]:
from haystack.nodes import DensePassageRetriever

dpr_gbert = DensePassageRetriever(
  document_store=document_store,
  query_embedding_model="deepset/gbert-base-germandpr-question_encoder",
  passage_embedding_model="deepset/gbert-base-germandpr-ctx_encoder",
  use_gpu=True,
  embed_title=True,
)

# Update embeddings
document_store.update_embeddings(dpr_gbert)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizerFast'.


Updating Embedding:   0%|          | 0/1211 [00:00<?, ? docs/s]

Create embeddings:   0%|          | 0/1216 [00:00<?, ? Docs/s]

In [17]:
from haystack.nodes import DensePassageRetriever

dpr_fb = DensePassageRetriever(
  document_store=document_store,
  query_embedding_model="facebook/dpr-question_encoder-single-nq-base",
  passage_embedding_model="facebook/dpr-ctx_encoder-single-nq-base",
  use_gpu=True,
  embed_title=True,

)

# facebook-dpr-question_encoder-multiset-base
# facebook-dpr-ctx_encoder-multiset-base

# Update embeddings
document_store.update_embeddings(dpr_fb)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizerFast'.
Documents Processed: 10000 docs [00:03, 2792.58 docs/s]       


### Embedding Retriever

In [1]:
from haystack.nodes import EmbeddingRetriever

embedding_retriever = EmbeddingRetriever(
    document_store=document_store,
    embedding_model="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
)

# Alternatives: https://www.sbert.net/docs/pretrained_models.html#

# distiluse-base-multilingual-cased-v1
#all-mpnet-base-v2
#paraphrase-multilingual-mpnet-base-v2
#all-distilroberta-v1

# Update embeddings
document_store.update_embeddings(embedding_retriever)

NameError: name 'document_store' is not defined

#### Reader

Here we use a FARMReader to initialize our reader.



##### FARMReader

In [31]:
from haystack.nodes import FARMReader
# Load a  local model or any of the QA models on
# Hugging Face's model hub (https://huggingface.co/models)
roberta_xlm_squad2 = FARMReader(model_name_or_path="deepset/xlm-roberta-large-squad2", use_gpu="mps")

In [21]:
from haystack.nodes import FARMReader
# Load a  local model or any of the QA models on
# Hugging Face's model hub (https://huggingface.co/models)
gelectra = FARMReader(model_name_or_path="deepset/gelectra-large-germanquad", use_gpu=True)

In [25]:
from haystack.nodes import FARMReader
# Load a  local model or any of the QA models on
# Hugging Face's model hub (https://huggingface.co/models)
roberta_squad2 = FARMReader(model_name_or_path="deepset/roberta-large-squad2", use_gpu=True)

### Pipeline

With a Haystack `Pipeline` you can stick together your building blocks to a search pipeline.
Under the hood, `Pipelines` are Directed Acyclic Graphs (DAGs) that you can easily customize for your own use cases.
To speed things up, Haystack also comes with a few predefined Pipelines. One of them is the `ExtractiveQAPipeline` that combines a retriever and a reader to answer our questions.
You can learn more about `Pipelines` in the [docs](https://docs.haystack.deepset.ai/docs/pipelines).

In [28]:
from haystack.pipelines import ExtractiveQAPipeline

pipe = ExtractiveQAPipeline(roberta_xlm_squad2, dpr_gbert)

# You can configure how many candidates the reader and retriever shall return
# The higher top_k for retriever, the better (but also the slower) your answers.
prediction = pipe.run(
    query="Wie lange darf man bei der Arbeit Pausen machen?", params={"Retriever": {"top_k": 10}, "Reader": {"top_k": 5}}
)

from haystack.utils import print_answers
print_answers(prediction, details="medium")

Inferencing Samples:   0%|          | 0/1 [00:00<?, ? Batches/s]

'Query: Wie lange darf man bei der Arbeit Pausen machen?'
'Answers:'
[   {   'answer': 'um die Mitte der Arbeitszeit',
        'context': 'en die gleichen Pausen wie für Erwachsene: Die Ar\xad\n'
                   'beit muss um die Mitte der Arbeitszeit durch Pausen von '
                   'folgender Min\xad\n'
                   'destdauer unterbrochen werde',
        'score': 0.7878586053848267},
    {   'answer': 'um die Mitte der Arbeitszeit',
        'context': 'en die gleichen Pausen wie für Erwachsene: Die Ar\xad\n'
                   'beit muss um die Mitte der Arbeitszeit durch Pausen von '
                   'folgender Min\xad\n'
                   'destdauer unterbrochen werde',
        'score': 0.6417407989501953},
    {   'answer': 'Min\xad\ndestdauer',
        'context': 'Mitte der Arbeitszeit durch Pausen von folgender Min\xad\n'
                   'destdauer unterbrochen werden:\n'
                   'Stunde bei einer Arbeitszeit von mehr als 51/2\n'
                 

In [43]:
from haystack.pipelines import ExtractiveQAPipeline

pipe = ExtractiveQAPipeline(gelectra, embedding_retriever)

# You can configure how many candidates the reader and retriever shall return
# The higher top_k for retriever, the better (but also the slower) your answers.
prediction = pipe.run(
    query="Ist es möglich, einem Lernenden wegen Krankheitstagen die Ferienzeit zu kürzen?", params={"Retriever": {"top_k": 10}, "Reader": {"top_k": 5}}
)

from haystack.utils import print_answers
print_answers(prediction, details="medium")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Inferencing Samples:   0%|          | 0/1 [00:00<?, ? Batches/s]

('Query: Ist es möglich, einem Lernenden wegen Krankheitstagen die Ferienzeit '
 'zu kürzen??')
'Answers:'
[   {   'answer': 'ist\nnicht möglich',
        'context': ' kein Lohn\n'
                   'geschuldet. Den Krankheitstag von den Ferien abzuziehen '
                   'ist\n'
                   'nicht möglich, da die Ferien zu Erholungszwecken dienen\n'
                   'und nicht „verrechne“ w',
        'score': 0.6089604496955872},
    {   'answer': 'ist\n'
                  'nicht möglich, da die Ferien zu Erholungszwecken dienen\n'
                  'und nicht „verrechne“ werden dürfen.',
        'context': ' von den Ferien abzuziehen ist\n'
                   'nicht möglich, da die Ferien zu Erholungszwecken dienen\n'
                   'und nicht „verrechne“ werden dürfen.\n'
                   'Beruf Polymechaniker\n'
                   'Stich',
        'score': 0.5453795194625854},
    {   'answer': 'Es gibt keine gesetzliche Regelung, die vorschreibt, wie '
        

In [48]:
from haystack.pipelines import ExtractiveQAPipeline

pipe = ExtractiveQAPipeline(gelectra, embedding_retriever)

# You can configure how many candidates the reader and retriever shall return
# The higher top_k for retriever, the better (but also the slower) your answers.
prediction = pipe.run(
    query="Mein Lohn wird immer verspätet überwiesen. Was kann ich tun?", params={"Retriever": {"top_k": 10}, "Reader": {"top_k": 5}}
)

from haystack.utils import print_answers
print_answers(prediction, details="medium")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Inferencing Samples:   0%|          | 0/1 [00:00<?, ? Batches/s]

'Query: Mein Lohn wird immer verspätet überwiesen. Was kann ich tun?'
'Answers:'
[   {   'answer': 'Vielleicht könnten Sie Ihrem Berufsbildner erklären,\n'
                  'dass Sie zuwenig Zeit haben für Ihre Aufgaben resp. für '
                  'die\n'
                  'Lerndokumentation, und Sie ihn deshalb bitten, Sie weniger\n'
                  'Überstunden leisten zu lassen resp. dass Sie diese abbauen\n'
                  'möchten.',
        'context': ' Vielleicht könnten Sie Ihrem Berufsbildner erklären,\n'
                   'dass Sie zuwenig Zeit haben für Ihre Aufgaben resp. für '
                   'die\n'
                   'Lerndokumentation, und Sie ihn deshalb bitten, Sie '
                   'weniger\n'
                   'Überstunden leisten zu lassen resp. dass Sie diese '
                   'abbauen\n'
                   'möchten.',
        'score': 0.18542206287384033},
    {   'answer': 'Antwort Grundsätzlich ist es so, dass eine Abmeldung nur '
           

In [51]:
from haystack.pipelines import ExtractiveQAPipeline

pipe = ExtractiveQAPipeline(gelectra, embedding_retriever)

# You can configure how many candidates the reader and retriever shall return
# The higher top_k for retriever, the better (but also the slower) your answers.
prediction = pipe.run(
    query="Wie viel Samstage muss ich arbeiten?", params={"Retriever": {"top_k": 10}, "Reader": {"top_k": 5}}
)

from haystack.utils import print_answers
print_answers(prediction, details="medium")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Inferencing Samples:   0%|          | 0/1 [00:00<?, ? Batches/s]

'Query: Wie viel Samstage muss ich arbeiten?'
'Answers:'
[   {   'answer': 'Am Sa wird durchgehend, ohne\nPause',
        'context': 'e Schwester eine einzige Pause\n'
                   'von nur gerade 15 Minuten. Am Sa wird durchgehend, ohne\n'
                   'Pause gearbeitet. Wenn man die Stunden pro Woche\n'
                   'ausrechnet erg',
        'score': 0.4044209420681},
    {   'answer': '2',
        'context': 'Bildung, MBA Bern\n'
                   '(für internen Gebrauch)\n'
                   'Bis jetzt habe ich nur 2 Samstage frei bekommen, weil sie '
                   'in\n'
                   'den Ferien lagen. Wenn ich an einem Samstag fre',
        'score': 0.38101881742477417},
    {   'answer': 'Kann mein Chef von mir verlangen, dass ich JEDEN\n'
                  'Samstag arbeite?\x0c'
                  'Chummerchaschte – Sammlig der Jahre 2009-2018\n'
                  'der Abteilung Betriebliche Bildung, MBA Bern\n'
                  '(für internen Geb